# 05 — Language Modeling Objectives

## Goal

A language model ultimately needs to predict the next token.

The Transformer produces a hidden representation for every token
position:

$$
H \in \mathbb{R}^{B \times T \times C}.
$$

However, the next token must be chosen from a vocabulary containing

$$
V
$$

possible tokens.

We therefore need to transform every hidden vector from dimension $C$
into $V$ output scores:

$$
(B,T,C)
\rightarrow
(B,T,V).
$$

These output scores are called **logits**.

The training objective then compares the logits with the true next-token
IDs and produces a scalar loss.

In [ ]:
import torch

batch_size: int = 2
context_length: int = 3
embedding_dim: int = 4
vocab_size: int = 6


hidden: torch.Tensor = torch.randn(
    batch_size,
    context_length,
    embedding_dim,
)

lm_head_weight: torch.Tensor = torch.randn(
    embedding_dim,
    vocab_size,
    requires_grad=True,
)

lm_head_bias: torch.Tensor = torch.zeros(
    vocab_size,
    requires_grad=True,
)

logits: torch.Tensor = hidden @ lm_head_weight + lm_head_bias

print("hidden:", hidden.shape)
print("logits:", logits.shape)

hidden: torch.Size([2, 3, 4])
logits: torch.Size([2, 3, 6])


## 2. Logits

A logit is an unconstrained real-valued score.

For one token position, the model produces

$$
z =
[z_1,z_2,\ldots,z_V].
$$

The logits:

- may be positive or negative,
- do not need to sum to one,
- are not probabilities.

Only the relative differences between logits matter when converting
them into probabilities.

Softmax transforms the logits into a probability distribution over the
vocabulary.

In [3]:
def softmax_stable(
    x: torch.Tensor,
    dim: int = -1,
) -> torch.Tensor:
    max_values: torch.Tensor = x.max(
        dim=dim,
        keepdim=True,
    ).values

    shifted: torch.Tensor = x - max_values
    exp_values: torch.Tensor = torch.exp(shifted)

    return exp_values / exp_values.sum(
        dim=dim,
        keepdim=True,
    )

In [ ]:
position_logits: torch.Tensor = logits[0, 0]
probabilities: torch.Tensor = softmax_stable(
    position_logits,
)

print("logits:", position_logits)
print("probabilities:", probabilities)
print("sum:", probabilities.sum())

logits: tensor([-0.1483, -0.8926, -1.2142,  0.9874, -1.2718, -1.1552],
       grad_fn=<SelectBackward0>)
probabilities: tensor([0.1778, 0.0845, 0.0612, 0.5536, 0.0578, 0.0650],
       grad_fn=<DivBackward0>)
sum: tensor(1., grad_fn=<SumBackward0>)


## 3. Negative Log-Likelihood

Suppose the correct next-token ID is $y$.

After softmax, the model assigns probability

$$
p_y
$$

to that token.

A good model should assign a high probability to the correct token.

The negative log-likelihood loss is

$$
L = -\log p_y.
$$

If

$$
p_y \rightarrow 1,
$$

then

$$
L \rightarrow 0.
$$

If

$$
p_y \rightarrow 0,
$$

then

$$
L \rightarrow \infty.
$$

The logarithm therefore strongly penalizes confident incorrect
predictions.

In [5]:
target_id: int = 2
correct_probability: torch.Tensor = probabilities[target_id]


loss: torch.Tensor = -torch.log(correct_probability)

print("correct probability:", correct_probability)
print("loss:", loss)

correct probability: tensor(0.0612, grad_fn=<SelectBackward0>)
loss: tensor(2.7928, grad_fn=<NegBackward0>)


## 4. From Softmax to Cross-Entropy

For one prediction position, let the model produce logits

$$
z =
[z_1, z_2, \ldots, z_V].
$$

Softmax converts the logits into probabilities:

$$
p_i
=
\frac{e^{z_i}}
{\sum_j e^{z_j}}.
$$

If the correct token ID is $y$, the negative log-likelihood is

$$
L
=
-\log p_y.
$$

Substituting the softmax definition gives

$$
L
=
-\log
\left(
\frac{e^{z_y}}
{\sum_j e^{z_j}}
\right).
$$

Using

$$
\log \frac{a}{b}
=
\log a - \log b,
$$

we obtain

$$
L
=
-z_y
+
\log
\sum_j e^{z_j}.
$$

This is the cross-entropy loss for a one-hot target.

Importantly, we can compute it directly from the logits without first
materializing the softmax probabilities.

In [10]:
# numerical stability: e^1002 -> overflow
logits = [1000, 1001, 1002]
torch.exp(torch.tensor(logits))

tensor([inf, inf, inf])

In [11]:
# so need to -1000 or logits max value
max_value = max(logits)

log_sum_exp = max_value + torch.log(
    torch.sum(torch.exp(torch.tensor(logits) - torch.tensor(max_value)))
)
log_sum_exp

tensor(1002.4076)

## 5. Stable Log-Sum-Exp

Directly evaluating

$$
\log \sum_j e^{z_j}
$$

can overflow when the logits are large.

Let

$$
m = \max_j z_j.
$$

Then

$$
\sum_j e^{z_j}
=
e^m
\sum_j e^{z_j-m}.
$$

Taking the logarithm gives

$$
\log\sum_j e^{z_j}
=
m
+
\log\sum_j e^{z_j-m}.
$$

Since every shifted value satisfies

$$
z_j - m \leq 0,
$$

the exponential terms remain numerically well behaved.

This is the same stability idea used in stable softmax.

In [7]:
def log_softmax_stable(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    max_values: torch.Tensor = x.max(dim=dim, keepdim=True).values

    shifted: torch.Tensor = x - max_values

    log_sum_exp: torch.Tensor = max_values + torch.log(
        torch.exp(shifted).sum(dim=dim, keepdim=True)
    )

    return x - log_sum_exp


example_logits: torch.Tensor = torch.tensor([1000.0, 1001.0, 1002.0])

log_probs: torch.Tensor = log_softmax_stable(example_logits)

print(log_probs)
print(torch.exp(log_probs))
print(torch.exp(log_probs).sum())

tensor([-2.4076, -1.4076, -0.4076])
tensor([0.0900, 0.2447, 0.6652])
tensor(1.0000)


In [8]:
def cross_entropy_single(logits: torch.Tensor, target_id: int) -> torch.Tensor:
    log_probs: torch.Tensor = log_softmax_stable(logits, dim=-1)

    return -log_probs[target_id]


logits: torch.Tensor = torch.tensor([1.2, -0.3, 2.0, 0.5])

target_id: int = 2

loss: torch.Tensor = cross_entropy_single(
    logits,
    target_id,
)

print(loss)

tensor(0.5725)


In [12]:
batch_size: int = 2
context_length: int = 3
vocab_size: int = 5

logits: torch.Tensor = torch.randn(
    batch_size,
    context_length,
    vocab_size,
)

targets: torch.Tensor = torch.tensor(
    [
        [1, 3, 2],
        [0, 4, 1],
    ],
    dtype=torch.long,
)

print(logits.shape)
print(targets.shape)

torch.Size([2, 3, 5])
torch.Size([2, 3])


In [13]:
# flatten (B,T)
flat_logits: torch.Tensor = logits.reshape(-1, vocab_size)
flat_targets: torch.Tensor = targets.reshape(-1)

print(flat_logits.shape)
print(flat_targets.shape)

torch.Size([6, 5])
torch.Size([6])


In [14]:
log_probs: torch.Tensor = log_softmax_stable(
    flat_logits,
    dim=-1,
)
row_indices: torch.Tensor = torch.arange(flat_targets.numel())

correct_log_probs: torch.Tensor = log_probs[
    row_indices,
    flat_targets,
]
loss: torch.Tensor = -correct_log_probs.mean()

print(loss)

tensor(2.0273)


In [16]:
def cross_entropy(
    logits: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    vocab_size: int = logits.shape[-1]

    flat_logits: torch.Tensor = logits.reshape(
        -1,
        vocab_size,
    )

    flat_targets: torch.Tensor = targets.reshape(-1)

    log_probs: torch.Tensor = log_softmax_stable(
        flat_logits,
        dim=-1,
    )

    row_indices: torch.Tensor = torch.arange(
        flat_targets.numel(),
        device=logits.device,
    )

    correct_log_probs: torch.Tensor = log_probs[
        row_indices,
        flat_targets,
    ]

    return -correct_log_probs.mean()

In [17]:
import torch.nn.functional as F


manual_loss: torch.Tensor = cross_entropy(
    logits,
    targets,
)

torch_loss: torch.Tensor = F.cross_entropy(
    logits.reshape(-1, vocab_size),
    targets.reshape(-1),
)

print("manual:", manual_loss)
print("PyTorch:", torch_loss)

print(
    torch.allclose(
        manual_loss,
        torch_loss,
    )
)

manual: tensor(2.0273)
PyTorch: tensor(2.0273)
True


In [18]:
def cross_entropy_with_padding(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pad_id: int,
) -> torch.Tensor:
    vocab_size: int = logits.shape[-1]

    flat_logits: torch.Tensor = logits.reshape(
        -1,
        vocab_size,
    )

    flat_targets: torch.Tensor = targets.reshape(-1)

    valid_mask: torch.Tensor = flat_targets != pad_id

    valid_logits: torch.Tensor = flat_logits[valid_mask]
    valid_targets: torch.Tensor = flat_targets[valid_mask]

    return cross_entropy(
        valid_logits,
        valid_targets,
    )

In [19]:
logits: torch.Tensor = torch.tensor(
    [1.0, 2.0, 3.0],
    requires_grad=True,
)

target_id: int = 2

loss: torch.Tensor = cross_entropy_single(
    logits,
    target_id,
)

loss.backward()

In [20]:
with torch.no_grad():
    probabilities: torch.Tensor = torch.softmax(
        logits,
        dim=-1,
    )

    expected_gradient: torch.Tensor = probabilities.clone()
    expected_gradient[target_id] -= 1.0

print("autograd:")
print(logits.grad)

print("expected:")
print(expected_gradient)

print(
    torch.allclose(
        logits.grad,
        expected_gradient,
    )
)

autograd:
tensor([ 0.0900,  0.2447, -0.3348])
expected:
tensor([ 0.0900,  0.2447, -0.3348])
True


## 13. A Useful Baseline: Uniform Predictions

Suppose the vocabulary contains $V$ tokens and the model has learned
nothing.

If it assigns equal probability to every token, then

$$
p_i = \frac{1}{V}.
$$

For the correct token, the negative log-likelihood is

$$
L
=
-\log\frac{1}{V}
=
\log V.
$$

Therefore,

$$
\boxed{L_{\text{uniform}} = \log V}
$$

provides a useful reference point.

For example, if

$$
V = 10,
$$

then a uniform predictor has loss

$$
\log 10 \approx 2.303.
$$

A language model that learns useful structure should eventually achieve
a validation loss below this uniform baseline.

In [21]:
import math

vocab_size: int = 10

uniform_loss: float = math.log(vocab_size)

print(uniform_loss)

2.302585092994046


## 14. Perplexity

Language-model cross-entropy is usually measured as the mean negative
log-likelihood per predicted token.

A related metric is **perplexity**:

$$
\mathrm{PPL}
=
\exp(L),
$$

where $L$ is the mean token-level cross-entropy.

Perplexity can be interpreted loosely as the effective number of tokens
among which the model is uncertain.

For a uniform predictor over a vocabulary of size $V$,

$$
L = \log V,
$$

so

$$
\mathrm{PPL}
=
e^{\log V}
=
V.
$$

Therefore, a uniform predictor over 10 tokens has perplexity 10.

Lower perplexity corresponds to lower cross-entropy and better
next-token prediction, provided that the tokenizer and evaluation setup
are held fixed.

In [22]:
def perplexity(loss: float) -> float:
    return math.exp(loss)


loss_value: float = math.log(10)
print(perplexity(loss_value))

10.000000000000002


## Takeaways

- A language model produces one vocabulary logit for every token
  position.

- Hidden states with shape

  $$
  (B,T,C)
  $$

  are projected into vocabulary logits with shape

  $$
  (B,T,V).
  $$

- Logits are unconstrained scores, not probabilities.

- Softmax converts logits into a probability distribution over the
  vocabulary.

- For a correct target token $y$, the negative log-likelihood is

  $$
  -\log p_y.
  $$

- For one-hot targets, negative log-likelihood and cross-entropy lead to

  $$
  L
  =
  -z_y
  +
  \log\sum_j e^{z_j}.
  $$

- Log-sum-exp should be computed with a numerical-stability shift.

- Language-model loss can be computed either directly on tensors of
  shape `(B, T, V)` or by flattening them to `(B*T, V)`.

- Flattening is a bookkeeping convenience and does not remove the
  contextual information already encoded in the logits.

- Padding targets should not contribute to the loss.

- For one prediction, the gradient of cross-entropy with respect to
  logit $z_j$ is

  $$
  \frac{\partial L}{\partial z_j}
  =
  p_j - \mathbf{1}[j=y].
  $$

- A uniform predictor over $V$ tokens has expected loss

  $$
  \log V.
  $$

- Perplexity is

  $$
  \mathrm{PPL}=e^L.
  $$